# Sanad v3 — Production vLLM Serving & Throughput/VRAM Benchmark (Colab L4)

Serves `Qwen/Qwen2.5-7B-Instruct` with our Milestone 1 fine-tuned LoRA adapter (`3isa2002/qwen2.5-7b-sanad-arabic-qlora`) behind an OpenAI-compatible **vLLM** server, benchmarks **TTFT**, **p50/p95 latency**, **output tokens/sec**, and **peak GPU VRAM**, and evaluates **Sanad v3 (`LangGraph + Qdrant + vLLM`)** on the 37-question golden set.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select Runtime -> Change runtime type -> L4 GPU"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | Total VRAM: {props.total_memory / (1024**3):.2f} GB")

In [ ]:
from pathlib import Path

REPO_DIR = Path("/content/sanad-support-rag")
if not REPO_DIR.exists():
    !git clone https://github.com/3isa2002/sanad-support-rag.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

%cd {REPO_DIR}
!pip install -q -e ".[dev]" vllm bitsandbytes

In [ ]:
import subprocess, time, httpx

cmd = [
    "vllm", "serve", "Qwen/Qwen2.5-7B-Instruct",
    "--host", "127.0.0.1", "--port", "8002",
    "--dtype", "half",
    "--quantization", "bitsandbytes",
    "--enable-lora",
    "--lora-modules", "sanad-qwen25-ft=3isa2002/qwen2.5-7b-sanad-arabic-qlora",
    "--max-model-len", "2048",
    "--gpu-memory-utilization", "0.90",
]
vllm_log = open("/content/vllm_server.log", "w")
vllm_proc = subprocess.Popen(cmd, stdout=vllm_log, stderr=subprocess.STDOUT)

for _ in range(90):
    if vllm_proc.poll() is not None:
        print(open("/content/vllm_server.log").read()[-3000:])
        raise RuntimeError("vLLM exited early")
    try:
        r = httpx.get("http://127.0.0.1:8002/v1/models", timeout=2)
        if r.status_code == 200:
            print("vLLM ready:", r.json())
            break
    except Exception:
        pass
    time.sleep(3)
else:
    print(open("/content/vllm_server.log").read()[-3000:])
    raise TimeoutError("Timed out waiting for vLLM")

In [ ]:
!python scripts/benchmark_vllm.py --base-url http://127.0.0.1:8002/v1 --model sanad-qwen25-ft --concurrency 1,4,8 --rounds 3 --output data/eval/vllm_benchmark.json
!nvidia-smi
print(open("data/eval/vllm_benchmark.json").read())

In [ ]:
import os
os.environ["SANAD_EMBEDDER"] = "hash"
os.environ["SANAD_LLM"] = "vllm"
os.environ["SANAD_VLLM_BASE_URL"] = "http://127.0.0.1:8002/v1"
os.environ["SANAD_VLLM_MODEL"] = "sanad-qwen25-ft"
os.environ["SANAD_TRACING"] = "jsonl"

!sanad ingest
!sanad eval -v
!sanad ask "مصاريف الشحن كام؟" --json
!sanad traces --limit 5

In [ ]:
vllm_proc.terminate()
vllm_proc.wait(timeout=15)
print("vLLM stopped.")